# CALFEM for Python - Introduction to Finite Element Analysis

**CALFEM** (Computer Aided Learning of the Finite Element Method) is an educational software package for teaching and learning the finite element method. This lecture covers the fundamentals of FEA using the Python implementation of CALFEM.

**CALFEM for Python** is a Python implementation of the CALFEM package, originally developed at Lund University since the late 1970s. It combines educational clarity with practical computing power:

- **Simplified Interface**: Clean Python API focusing on the core FEM concepts rather than implementation details
- **NumPy Integration**: Uses NumPy matrices for efficient scientific computing
- **Transparent**: Each step of the FE process is explicit and transparent - you control assembly, solving, and post-processing
- **Multiple Element Types**: Supports springs, bars, 2D elements, plates, and 3D elements
- **Visualization Tools**: Built-in plotting with Matplotlib for geometry, meshes, and results
- **Extensible**: Full Python integration allows custom development and problem-solving

## Key Technologies
- **NumPy**: For efficient matrix operations and numerical computing
- **Matplotlib**: For visualization of geometries, meshes, and results
- **Optional Mesh Generators**: Integration with Triangle and GMSH for complex geometries

## Learning Objectives

By the end of this lecture, you will be able to:
- Understand the basic workflow of finite element analysis
- Use CALFEM to model and solve structural mechanics problems
- Assemble element matrices into global system matrices
- Apply boundary conditions and solve for displacements
- Calculate element forces and verify results
- Work with different element types (springs, bars, frames)

## Reference Materials
- **Official Documentation**: [CALFEM Python Manual](https://calfem-python-manual.readthedocs.io/en/latest/)
- **GitHub Repository**: [CALFEM for Python](https://github.com/CALFEM/calfem-python/)
- **Examples**: [CALFEM Examples Collection](https://calfem-python-manual.readthedocs.io/en/latest/examples.html)

### What Makes CALFEM Unique for Teaching FEA?



---
# CALFEM Python modules

* **calfem.core**
  * Element routines
  * System routines
* **calfem.utils**
  * I/O routines
  * Misc. routines
* **calfem.geometry**
  * Routines for defining problem geometry used for input in mesh generation
* **calfem.mesh**
  * Mesh generation routines
* **calfem.vis/calfem.vis_mpl**
  * Routines for visualising geometry, meshes and results.

---
# Installing CALFEM for Pyton

In [8]:
!pip install calfem-python

---
# General procedure for FE calculation

### The Finite Element Method Workflow

Every finite element analysis follows a systematic process. Understanding each step is crucial for successful problem-solving:

1. **Define the Model**
   - Establish the geometry and problem domain
   - Define material properties (Young's modulus, cross-sectional area, etc.)
   - Identify boundary conditions and loads
   - Create the element topology (which nodes form each element)

2. **Generate Element Matrices**
   - For each element, compute the local stiffness matrix using element equations
   - In CALFEM, this is done using element functions like `spring1e()`, `bar2e()`, `beam2e()`, etc.
   - Element matrices relate local displacements to local forces

3. **Assemble Global System**
   - Combine all element matrices into a single global stiffness matrix **K**
   - This assembly maps local DOFs to global DOFs using the topology matrix
   - The global matrix represents the entire structure's resistance to deformation

4. **Apply Boundary Conditions & Solve**
   - Specify known displacements (supports, fixed boundaries)
   - Apply external loads/forces to the load vector
   - Solve the global system: **K·a = f** for unknown displacements **a**

5. **Calculate Element Results**
   - Extract element displacements from global solution
   - Compute element stresses, strains, and internal forces
   - Verify solution through reaction forces and equilibrium checks

---
# Example 1 - System of springs

### Problem Description

**Reference**: [exs_spring](https://calfem-python-manual.readthedocs.io/en/latest/examples.html#exs-spring) in the CALFEM manual.

![Spring system — physical structure](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs1_1.svg)
![Spring system — finite element model](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs1_2.svg)

**Three springs in series** with different stiffness properties:
- Node 1 (left end): Fixed support (displacement = 0)
- Spring 1: Stiffness k₁ = 2k = 3000 N/m, connects nodes 1–2
- Spring 2: Stiffness k₂ = k = 1500 N/m, connects nodes 2–3 (parallel with spring 3)
- Spring 3: Stiffness k₃ = 2k = 3000 N/m, connects nodes 2–3 (parallel with spring 2)
- Node 3 (right end): Fixed support (displacement = 0)
- **Applied load**: F = 100 N at node 2

**Learning Goals**:
- Understand the topology matrix `Edof`
- Learn how to assemble element stiffness matrices into the global system
- Solve a constrained system of equations
- Extract and interpret element forces

First we import the required Python-modules:

In [9]:
import numpy as np
import calfem.core as cfc
import calfem.vis_mpl as cfv
import math

### Step 1: Define Element Topology

The **Edof** (Element degrees of freedom) matrix defines which global DOFs belong to each element. It maps element connectivity to the global coordinate system.

**Matrix Interpretation**:
- Each row represents one element
- Columns contain the global DOF numbers for that element
- For 1D spring elements: [start_node, end_node]

**Important Note**: CALFEM uses **1-based indexing** for DOFs (not Python's 0-based indexing). DOF 1 corresponds to the displacement of node 1, etc.

**In our problem**:
- Row 0: Element connecting nodes 1-2 → DOFs [1, 2]
- Row 1: Element connecting nodes 2-3 → DOFs [2, 3]  
- Row 2: Element connecting nodes 2-3 → DOFs [2, 3] (parallel spring)

In [10]:
Edof = np.array([
        [1,2],
        [2,3],
        [2,3]
    ])
print(Edof)

[[1 2]
 [2 3]
 [2 3]]


Stiffness matrix and force vector.

In [11]:
K = np.zeros((3,3))
f = np.zeros((3,1))
print(K)
print(f)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[0.]
 [0.]
 [0.]]


### Step 2: Generate Element Stiffness Matrices

For each spring element, CALFEM computes the local stiffness matrix. For a 1D spring with stiffness k:

$$K_e = \begin{bmatrix} k & -k \\ -k & k \end{bmatrix}$$

**Function**: `cfc.spring1e(ep)` where `ep` is the element property (spring stiffness k)

**Input Parameters**:
- `ep`: Spring constant (N/m)

**Output**:
- `Ke`: 2×2 stiffness matrix relating forces to displacements in the local coordinate system

**Physical Interpretation**:
- Diagonal terms: self-resistance (diagonal terms always positive)
- Off-diagonal terms: coupling between nodes (always negative, symmetric)

In [12]:
k = 1500.0
ep1 = k
ep2 = 2.0*k
Ke1 = cfc.spring1e(ep1)
Ke2 = cfc.spring1e(ep2)
print(Ke1)
print(Ke2)

[[ 1500. -1500.]
 [-1500.  1500.]]
[[ 3000. -3000.]
 [-3000.  3000.]]


### Step 3: Assemble Element Matrices into Global System

**Assembly** is the process of combining all element contributions into a global stiffness matrix **K** and global load vector **f**.

**How it works**:
- Each element matrix (2×2 for springs) is placed into the global matrix (3×3 for 3 nodes)
- The position is determined by the topology information in the Edof matrix
- Contributions are **added** where DOFs overlap (this enforces equilibrium at shared nodes)

**CALFEM Function**: `cfc.assem(Edof_row, K_global, Ke_element)`

**The Global Stiffness Matrix** represents the entire system's behavior:
- Dimension: (n_dofs × n_dofs), in our case 3×3
- Symmetric and positive semi-definite
- Reflects how the structure resists deformation

**Note on DOF ordering**: 
- Row/Column 0 corresponds to DOF 1 (node 1)
- Row/Column 1 corresponds to DOF 2 (node 2)
- Row/Column 2 corresponds to DOF 3 (node 3)

In [13]:
cfc.assem(Edof[0,:], K, Ke2)
cfc.assem(Edof[1,:], K, Ke1)
cfc.assem(Edof[2,:], K, Ke2)

print(K)

[[ 3000. -3000.     0.]
 [-3000.  7500. -4500.]
 [    0. -4500.  4500.]]


### Step 4: Solve the System of Equations

**Boundary Conditions** specify constraints on the system:
- **Prescribed DOFs**: Nodes with known displacements (typically supports/fixed nodes)
- **Free DOFs**: Nodes where displacement is unknown (to be solved for)

**Array Indexing Clarification**: 
Remember that `bc` array uses **1-based DOF numbering**, but when passed to NumPy arrays it's internally converted.
- `bc = np.array([1, 3])` means DOFs 1 and 3 are prescribed (nodes 1 and 3 are fixed)
- `f[1] = 100.0` applies a 100 N load at DOF 2 (note: Python 0-based indexing!)

**Solving**: `cfc.solveq(K, f, bc)`
Solves the system: **K·a = f** subject to prescribed displacements

**Output**:
- `a`: Displacements at all DOFs (includes prescribed zero values and solved unknowns)
- `r`: Reaction forces at prescribed DOFs (forces needed to maintain the boundary conditions)

In [14]:
bc = np.array([1,3])

f[1] = 100.0

# Solve equation system

a, r = cfc.solveq(K, f, bc)

print("Displacements:")
print(a)
print("Reaction forces:")
print(r)

Displacements:
[[0.        ]
 [0.01333333]
 [0.        ]]
Reaction forces:
[[-40.]
 [  0.]
 [-60.]]


### Step 5: Post-Processing - Extract and Calculate Element Results

**Post-processing** verifies the solution and computes quantities of interest:

1. **Extract element displacements**: `cfc.extract_eldisp(edof_row, a)`
   - Takes the global displacement vector and extracts the displacements for a specific element
   
2. **Calculate element forces**: `cfc.spring1s(ep, ed)`
   - Computes the internal force (spring force) in each element
   - Formula: N = k·(u_end - u_start)
   - Positive value = tension, negative = compression

**Verification**:
- Check that reaction forces sum to applied loads (global equilibrium)
- Check that element forces are consistent with element displacements

In [15]:
ed1 = cfc.extract_eldisp(Edof[0,:], a)
ed2 = cfc.extract_eldisp(Edof[1,:], a)
ed3 = cfc.extract_eldisp(Edof[2,:], a)

es1 = cfc.spring1s(ep2, ed1)
es2 = cfc.spring1s(ep1, ed2)
es3 = cfc.spring1s(ep2, ed3)

In [16]:
print("N1 =", es1)
print("N2 =", es2)
print("N3 =", es3)

N1 = 40.0
N2 = -20.0
N3 = -40.0


### Example 1 Summary and Verification

**Results Interpretation**:
- Displacements show how much each node moved under the applied load
- Reaction forces at fixed supports should sum to applied load (equilibrium check)
- Element forces show the internal stress in each spring

**Verification Checks**:
1. **Global Equilibrium**: Sum reaction forces = Applied load
2. **Element State**: Positive force = tension, negative = compression
3. **Displacement Consistency**: Larger spring constants should have smaller displacements

**Key Learning Points**:
- The topology matrix (Edof) is the critical link between elements and global system
- Assembly combines local information into global system
- Boundary conditions reduce the number of unknowns
- Element forces are calculated from displacements using constitutive relations

---
# Example 2 - Bars

### Example 2: 2D Truss Analysis — Three Bars

**Reference**: [exs_bar2](https://calfem-python-manual.readthedocs.io/en/latest/examples.html#exs-bar2) in the CALFEM manual.

![Three-bar truss — geometry and finite element model](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs3.svg)

**Problem Description**:
Three bar elements with different cross-sectional areas, loaded by a single downward force P = 80 kN:
- E = 200 GPa for all bars
- A₁ = 6.0×10⁻⁴ m², A₂ = 3.0×10⁻⁴ m², A₃ = 10.0×10⁻⁴ m²
- Fixed supports at nodes 1, 2, and 4
- 8 degrees of freedom total (4 nodes × 2 DOFs each)

**Learning Objectives**:
- Work with 2D elements requiring coordinate information (ex, ey)
- Understand local-to-global coordinate transformation in bar elements
- Handle multiple DOFs per node (u_x and u_y)
- Interpret normal force results (tension/compression)

**Key Differences from Springs**:
- Bars use material modulus E and cross-section area A instead of spring constant k
- Elements exist in 2D space, requiring end-node coordinates
- Each node has 2 DOFs (horizontal and vertical displacement)

### Element Topology for 2D Elements

For 2D bar elements, the **Edof** matrix contains 4 entries per row:
- **[node1_u_x, node1_u_y, node2_u_x, node2_u_y]**

This maps the 2 nodes of each bar element to their 4 global DOFs.

**DOF Numbering Convention** in CALFEM for 2D problems:
- DOF 1, 2: u_x, u_y at node 1
- DOF 3, 4: u_x, u_y at node 2
- DOF 5, 6: u_x, u_y at node 3
- DOF 7, 8: u_x, u_y at node 4
- etc.

**Our Truss Elements**:
- Element 1: connects nodes 1-3 → DOFs [1,2,5,6]
- Element 2: connects nodes 3-4 → DOFs [5,6,7,8]
- Element 3: connects nodes 2-3 → DOFs [3,4,5,6]

In [17]:
Edof = np.array([
    [1,2,5,6],
    [5,6,7,8],
    [3,4,5,6]
])

Stiffness matrix and load vector.

In [18]:
K = np.zeros((8,8))
f = np.zeros((8,1))

### Material and Geometric Properties

For 2D bar elements, we need to specify:
- **E**: Young's modulus (material property) [Pa]
- **A**: Cross-sectional area [m²]
- **L**: Length (automatically computed from coordinates)

**Element Property Vector** `ep`:
- For CALFEM bar elements: `ep = [E, A]`

**Our Problem**:
- Material: Steel with E = 2.0×10¹¹ Pa
- Three bars with different areas:
  - A₁ = 6.0×10⁻⁴ m² (main diagonal bar)
  - A₂ = 3.0×10⁻⁴ m² (secondary diagonal)
  - A₃ = 10.0×10⁻⁴ m² (horizontal bar)

In [19]:
E = 2.0e11
A1 = 6.0e-4
A2 = 3.0e-4
A3 = 10.0e-4
ep1 = [E, A1] # Vanliga listor kan användas för att definiera element-
ep2 = [E, A2] # egenskaper
ep3 = [E, A3]

### Element Geometry - Nodal Coordinates

For 2D bar elements, we need the **x and y coordinates** of each element's endpoints:

**Input Format**:
- `ex`: Array of x-coordinates for element endpoints [x_start, x_end]
- `ey`: Array of y-coordinates for element endpoints [y_start, y_end]

**Purpose**:
- CALFEM automatically computes element length: L = √[(Δx)² + (Δy)²]
- Computes orientation angle for local-to-global transformations
- Orientation is critical for 2D elements (springs only care about length, but bars care about angle)

**Our Truss Geometry**:
- Element 1 (ex1, ey1): Horizontal bar from (0,0) to (1.6 m, 0 m)
- Element 2 (ex2, ey2): Vertical bar from (1.6 m, 0 m) to (1.6 m, 1.2 m)
- Element 3 (ex3, ey3): Diagonal bar from (0 m, 1.2 m) to (1.6 m, 0 m)

In [20]:
ex1=np.array([0.,1.6])
ex2=np.array([1.6,1.6])
ex3=np.array([0.,1.6])

ey1=np.array([0.,0.])
ey2=np.array([0.,1.2])
ey3=np.array([1.2,0.])

### Generate Element Stiffness Matrices for 2D Bars

**CALFEM Function**: `cfc.bar2e(ex, ey, ep)`

**Input Parameters**:
- `ex`: x-coordinates of element endpoints [x_start, x_end]
- `ey`: y-coordinates of element endpoints [y_start, y_end]  
- `ep`: element properties [E, A] (Young's modulus and cross-sectional area)

**What CALFEM Computes Internally**:
1. Element length: L = √[(x_end - x_start)² + (y_end - y_start)²]
2. Element orientation angle: θ = atan2(y_end - y_start, x_end - x_start)
3. Local stiffness matrix (for element aligned with its own axis)
4. **Transformation matrix** to rotate from local to global coordinates
5. Global stiffness matrix (4×4 for 2D bar element)

**Output**: 
- `Ke`: 4×4 stiffness matrix in global coordinate system
- Relates global DOFs [u_x1, u_y1, u_x2, u_y2] to global forces

In [21]:
Ke1 = cfc.bar2e(ex1, ey1, ep1)
Ke2 = cfc.bar2e(ex2, ey2, ep2)
Ke3 = cfc.bar2e(ex3, ey3, ep3)

Assemble stiffness matrix.

In [22]:
cfc.assem(Edof[0,:],K,Ke1)
cfc.assem(Edof[1,:],K,Ke2)
cfc.assem(Edof[2,:],K,Ke3)

array([[ 7.50e+07,  0.00e+00,  0.00e+00,  0.00e+00, -7.50e+07,  0.00e+00,
         0.00e+00,  0.00e+00],
       [ 0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,
         0.00e+00,  0.00e+00],
       [ 0.00e+00,  0.00e+00,  6.40e+07, -4.80e+07, -6.40e+07,  4.80e+07,
         0.00e+00,  0.00e+00],
       [ 0.00e+00,  0.00e+00, -4.80e+07,  3.60e+07,  4.80e+07, -3.60e+07,
         0.00e+00,  0.00e+00],
       [-7.50e+07,  0.00e+00, -6.40e+07,  4.80e+07,  1.39e+08, -4.80e+07,
         0.00e+00,  0.00e+00],
       [ 0.00e+00,  0.00e+00,  4.80e+07, -3.60e+07, -4.80e+07,  8.60e+07,
         0.00e+00, -5.00e+07],
       [ 0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,
         0.00e+00,  0.00e+00],
       [ 0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00,  0.00e+00, -5.00e+07,
         0.00e+00,  5.00e+07]])

### Solve the 2D System

**Boundary Conditions** (using 1-based DOF indexing):
- bc = [1,2,3,4]: DOFs 1,2,3,4 are fixed (all x,y displacements at nodes 1 and 2 are zero)
- This means nodes 1 and 2 are fixed supports

**Applied Load**:
- f[5] = -80e3: 80,000 N downward (negative y-direction) applied to DOF 5 (u_y at node 3)

**Solution Process**:
1. Remove equations corresponding to prescribed DOFs (1,2,3,4)
2. Solve reduced system for free DOFs (5,6,7,8)
3. Back-substitute to find reaction forces at prescribed DOFs

In [23]:
bc = np.array([1,2,3,4,7,8])
f[5] = -80e3

a, r = cfc.solveq(K, f, bc)

print("Displacements:")
print(a)
print("Reaction forces:")
print(r)

Displacements:
[[ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [-0.00039793]
 [-0.00115233]
 [ 0.        ]
 [ 0.        ]]
Reaction forces:
[[ 29844.55958549]
 [     0.        ]
 [-29844.55958549]
 [ 22383.41968912]
 [     0.        ]
 [     0.        ]
 [     0.        ]
 [ 57616.58031088]]


### Post-Processing: Calculate Bar Forces

**Bar Element Force Computation**:

**CALFEM Function**: `cfc.bar2s(ex, ey, ep, ed)`

**Inputs**:
- `ex, ey`: Element coordinates (same as used in bar2e)
- `ep`: Element properties [E, A]
- `ed`: Element displacements (extracted from global solution)

**Output**:
- Axial force N in the bar element
- Positive = tension, Negative = compression

**Physical Interpretation**:
- Axial stress: σ = N / A
- Axial strain: ε = ΔL / L
- Internal force equilibrium verifies: local element forces match nodal forces

In [24]:
ed1=cfc.extract_eldisp(Edof[0,:],a);
N1=cfc.bar2s(ex1,ey1,ep1,ed1)
ed2=cfc.extract_eldisp(Edof[1,:],a);
N2=cfc.bar2s(ex2,ey2,ep2,ed2)
ed3=cfc.extract_eldisp(Edof[2,:],a);
N3=cfc.bar2s(ex3,ey3,ep3,ed3)
print("N1=",N1)
print("N2=",N2)
print("N3=",N3)

N1= [[-29844.55958549]
 [-29844.55958549]]
N2= [[57616.58031088]
 [57616.58031088]]
N3= [[37305.69948187]
 [37305.69948187]]


# Example 3 - Complex Truss Structure

**A More Realistic Problem**: 10-Bar Planar Truss with Realistic Geometry

### Problem Setup

**Reference**: [exs_bar2_l](https://calfem-python-manual.readthedocs.io/en/latest/examples.html#exs-bar2-l) in the CALFEM manual.

![10-bar truss — physical structure and loading](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs4_1.svg)
![10-bar truss — finite element model with node and DOF numbering](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs4_2.svg)

**Structure**: A 10-bar planar truss with:
- 6 nodes arranged in a 3×2 rectangular grid
- 10 bar elements (horizontal, vertical, and diagonal members)
- Fixed supports at nodes 1 and 2 (bottom-left)
- Applied load P = 0.5 MN at node 6 at 30° from vertical

**Material properties** (uniform):
- E = 2.1×10¹¹ Pa, A = 25×10⁻⁴ m²
- ep = [E, A] — same for all elements

**Why loops?** With 10 elements, manual assembly would be tedious and error-prone. This example introduces loop-based assembly — the standard approach for all practical FE programs.

### Step 1: Define Element Topology with 10 Elements

The **Edof** matrix has 10 rows (one for each bar element) and 4 columns (2 nodes, 2 DOFs each).

**Truss Configuration** (each row shows which nodes are connected):
- Elements 1-4: Connect main grid points vertically and horizontally
- Elements 5-6: Connect vertical members
- Elements 7-8: Connect horizontal members  
- Elements 9-10: Diagonal bracing members (provide lateral stiffness)

**DOF Numbering for 12 DOFs total**:
- Node 1: DOFs 1, 2 (u_x, u_y at bottom-left)
- Node 2: DOFs 3, 4 (u_x, u_y at bottom-middle)
- Node 3: DOFs 5, 6 (u_x, u_y at bottom-right)
- Node 4: DOFs 7, 8 (u_x, u_y at top-left)
- Node 5: DOFs 9, 10 (u_x, u_y at top-middle)
- Node 6: DOFs 11, 12 (u_x, u_y at top-right)

In [25]:
Edof = np.array([
    [1, 2, 5, 6],
    [3, 4, 7, 8],
    [5, 6, 9, 10],
    [7, 8, 11, 12],
    [7, 8, 5, 6],
    [11, 12, 9, 10],
    [3, 4, 5, 6],
    [7, 8, 9, 10],
    [1, 2, 7, 8],
    [5, 6, 11, 12]
])

Stiffness matrix and element properties.

In [26]:
K=np.zeros([12,12])
f=np.zeros([12,1])

A = 25.0e-4
E = 2.1e11
ep = [E,A]

### Step 2: Define Nodal Coordinates for All Elements

**Coordinate Arrays**:
- `ex`: Array of size (10, 2) with x-coordinates for each element's two endpoints
- `ey`: Array of size (10, 2) with y-coordinates for each element's two endpoints

**Coordinate System** (in meters):
- Bottom-left (node 1) at (0, 0)
- Bottom-middle (node 2) at (2, 0)
- Bottom-right (node 3) at (4, 0)
- Top-left (node 4) at (0, 2)
- Top-middle (node 5) at (2, 2)
- Top-right (node 6) at (4, 2)

**Element Geometry Mapping**:
- Horizontal members: Same y-coordinate at both ends
- Vertical members: Same x-coordinate at both ends
- Diagonal members: Both x and y change between endpoints

In [27]:
ex = np.array([
    [0., 2.],
    [0., 2.],
    [2., 4.],
    [2., 4.],
    [2., 2.],
    [4., 4.],
    [0., 2.],
    [2., 4.],
    [0., 2.],
    [2., 4.]
])

ey = np.array([
    [2., 2.],
    [0., 0.],
    [2., 2.],
    [0., 0.],
    [0., 2.],
    [0., 2.],
    [0., 2.],
    [0., 2.],
    [2., 0.],
    [2., 0.]
])


### Step 3: Efficient Assembly Using Loops

**Why Loops?** For large structures with many elements, manual assembly is impractical. Loops automate the process.

**Loop Strategy**: 
- Iterate over all elements
- For each element: extract coordinates (ex line, ey line) and topology (Edof row)
- Compute stiffness matrix using `bar2e()`
- Assemble into global matrix using `assem()`

**Python `zip()` Function**:
- Allows simultaneous iteration over multiple arrays
- `zip(ex, ey, Edof)` yields (ex_row, ey_row, edof_row) tuples one at a time
- Elegant and Pythonic way to iterate element-by-element

In [28]:
for elx, ely, eltopo in zip(ex, ey, Edof):
    Ke = cfc.bar2e(elx, ely, ep)
    cfc.assem(eltopo, K, Ke)

### Step 4: Apply Boundary Conditions and Solve

Fix all four nodes of the support structure (nodes 1–4 each have 2 DOFs):
- `bc = [1, 2, 3, 4]` — DOFs 1 and 2 (node 1 at origin), DOFs 3 and 4 (node 2)

Apply the external force at node 6 (DOF 10 = x, DOF 11 = y), resolved into components:
- A 0.5 MN load at 30° from vertical: $F_x = 0.5\sin(30°)$, $F_y = -0.5\cos(30°)$

In [29]:
bc = np.array([1,2,3,4])

f[10]=0.5e6*math.sin(math.pi/6)
f[11]=-0.5e6*math.cos(math.pi/6)

a, r = cfc.solveq(K,f,bc)

### Step 5: Compute and Print Element Forces

`cfc.bar2s(ex, ey, ep, ed)` returns the scalar axial force $N$ in each bar element:
- **Positive** N → element in **tension**
- **Negative** N → element in **compression**

All element displacements are extracted at once with `cfc.extract_eldisp(Edof, a)`, then iterated in the same loop pattern used for assembly.

In [ ]:
ed = cfc.extract_eldisp(Edof, a)  # Extract all element displacements

N = np.zeros([Edof.shape[0]])  # Array to store element forces

print("Element forces:")

i = 0
for elx, ely, eld in zip(ex, ey, ed):

    es = cfc.bar2s(elx, ely, ep, eld)
    N[i] = es[0].item()  # Extract scalar from a 1 x 1 array
    print("N%d = %g" % (i+1, N[i]))
    i += 1

Element forces:
N1 = 625938
N2 = -423100
N3 = 170640
N4 = -12372.8
N5 = -69447
N6 = 170640
N7 = -272838
N8 = -241321
N9 = 339534
N10 = 371051


---
# Example 6 — Mixed Frame and Bar Elements

## Learning Objectives
- Understand **beam elements** (`beam2e`) with 3 DOFs per node (u, v, θ)
- Combine different element types in a single model
- Use the `coordxtr()` helper to extract coordinates from node/DOF tables
- Understand the role of `dof1` (beam) and `dof2` (bar) DOF maps

## The Mixed-Element Concept

Real structures often contain members that behave very differently:

| Element type | function | DOFs/node | Resists |
|---|---|---|---|
| **Beam** (`beam2e`) | Bending + axial | 3 (u, v, θ) | Axial force, shear, moment |
| **Bar** (`bar2e`) | Axial only | 2 (u, v) | Axial force only |

This example builds a **two-bay portal frame** with 6 nodes (18 total DOFs), where:
- **6 beam elements** form the columns and horizontal beams
- **4 bar elements** form diagonal and horizontal cross-bracings

Because beam and bar elements share the same **translational** DOFs (u, v) but beams also have a **rotation** DOF (θ), two separate DOF tables (`dof1` for beams, `dof2` for bars) are used.

**Reference**: [exs_beam2](https://calfem-python-manual.readthedocs.io/en/latest/examples.html#exs-beam2) (plane frame) and [exs_beambar2](https://calfem-python-manual.readthedocs.io/en/latest/examples.html#exs-beambar2) (combined beam and bar) in the CALFEM manual.

The figures below show the reference frame structure from the manual. In this notebook's version, the frame is extended with bar elements acting as diagonal and horizontal cross-bracings, demonstrating how beam and bar elements share the same global DOF system.

![Plane frame — physical structure](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs6_1.svg)
![Combined beam and bar — problem definition](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs7_1.svg)
![Combined beam and bar — finite element model](https://calfem-python-manual.readthedocs.io/en/latest/_images/exs7_2.svg)

### Step 1: Define System Matrices and Geometry

**System size**: 6 nodes × 3 DOFs/node = **18 DOFs total** → `K` is 18×18.

**Node coordinates** (`coord`): Six nodes arranged in a 2×3 grid (width×height = 1×2):

| Node | x | y | Location |
|---|---|---|---|
| 1 | 0 | 0 | Bottom-left |
| 2 | 1 | 0 | Bottom-right |
| 3 | 0 | 1 | Mid-left |
| 4 | 1 | 1 | Mid-right |
| 5 | 0 | 2 | Top-left |
| 6 | 1 | 2 | Top-right |

**DOF tables**: Used by `coordxtr()` to locate each node's DOFs.
- `dof1` — 3 DOFs per node `[u, v, θ]` → for beam elements: DOFs 1–3, 4–6, 7–9, …
- `dof2` — 2 DOFs per node `[u, v]` → for bar elements: first two DOFs from each node

**Load**: `f[12, 0] = 1.0` → unit horizontal force at DOF 13 (u of node 5, top-left).

In [31]:
K = np.zeros([18,18])
f = np.zeros([18,1])
f[12,0] = 1.0

coord = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 2.0],
    [1.0, 2.0]
])

dof1 = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12],
    [13, 14, 15],
    [16, 17, 18]
])

dof2 = np.array([
    [1, 2],
    [4, 5],
    [7, 8],
    [10, 11],
    [13, 14],
    [16, 17]
])

### Step 2: Element Properties, Topology, and Coordinates

**Beam element properties** `ep1 = [E, A, I]`:
- `E` = Young's modulus
- `A` = cross-sectional area
- `I` = second moment of area (bending stiffness)

**Bar element properties** `ep2 = [E, A]`:
- Bars cannot resist bending, so I is not needed

**Connectivity matrices** (`edof1` for beams, `edof2` for bars):
- Beam elements: 6 DOFs each (2 nodes × 3 DOF)
- Bar elements: 4 DOFs each (2 nodes × 2 DOF)

**`cfc.coordxtr(edof, coord, dof)`** extracts coordinate arrays `(ex, ey)` for all elements:
- Takes the DOF connectivity table, node coordinate array, and node-DOF table
- Returns row arrays where each row holds the coordinates of an element's end nodes
- Replaces the manual `ex`/`ey` construction used in earlier examples

In [32]:
ep1 = [1.0, 1.0, 1.0]

edof1 = np.array([
    [1, 2, 3, 7, 8, 9],
    [7, 8, 9, 13, 14, 15],
    [4, 5, 6, 10, 11, 12],
    [10, 11, 12, 16, 17, 18],
    [7, 8, 9, 10, 11, 12],
    [13, 14, 15, 16, 17, 18]
])

ex1, ey1 = cfc.coordxtr(edof1, coord, dof1);

ep2 = [1.0, 1.0]

edof2 = np.array([
    [1, 2, 10, 11],
    [7, 8, 16, 17],
    [7, 8, 4, 5],
    [13, 14, 10, 11]
])

ex2, ey2 = cfc.coordxtr(edof2, coord, dof2);

### Step 3: Assemble Beam and Bar Element Stiffness Matrices

The two element types are assembled in **separate loops** into the same global `K`:

**Beam loop** — uses `cfc.beam2e(ex, ey, ep1)`:
- Returns a **6×6** element stiffness matrix in global coordinates
- Each row/column corresponds to one beam-element DOF (u₁, v₁, θ₁, u₂, v₂, θ₂)

**Bar loop** — uses `cfc.bar2e(ex, ey, ep2)`:
- Returns a **4×4** element stiffness matrix in global coordinates
- Only translational DOFs are used (u₁, v₁, u₂, v₂), rotation not included

Both loops call `cfc.assem(eltopo, K, Ke)` to scatter the element contributions into the shared 18×18 global matrix. The DOF indices in `eltopo` ensure bar DOFs (e.g., 1, 2) land in the correct rows and columns alongside the beam DOFs.

In [33]:
for elx, ely, eltopo in zip(ex1, ey1, edof1):
    Ke = cfc.beam2e(elx, ely, ep1)
    cfc.assem(eltopo, K, Ke)

In [34]:
for elx, ely, eltopo in zip(ex2, ey2, edof2):
    Ke = cfc.bar2e(elx, ely, ep2)
    cfc.assem(eltopo,K,Ke)

### Step 4: Apply Boundary Conditions and Solve

Fix both base nodes (nodes 1 and 2) fully — all 6 DOFs (u, v, θ at each):
```
bc_prescr = [1, 2, 3, 4, 5, 6]
```
This represents a **clamped-base portal frame**: no translation or rotation permitted at the supports.

`cfc.solveq(K, f, bc_prescr)` returns:
- `a` — displacement vector (18×1): translational + rotational displacements at all free nodes
- `r` — reaction vector (18×1): support reactions at the clamped base nodes

In [35]:
bc_prescr = np.array([1, 2, 3, 4, 5, 6])
a, r = cfc.solveq(K, f, bc_prescr)

In [36]:
print(a)

[[ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 0.37905924]
 [ 0.30451926]
 [-0.65956297]
 [ 0.3041448 ]
 [-0.28495132]
 [-0.54570174]
 [ 1.19791809]
 [ 0.44655174]
 [-0.85908643]
 [ 0.96969909]
 [-0.34780417]
 [-0.74373562]]


In [ ]:
print("=== Example 6 Displacement Results ===")
print(f"Number of DOFs: {a.shape[0]}")

# Print displacements grouped by node
node_labels = ["Node 1 (0,0)", "Node 2 (1,0)", "Node 3 (0,1)",
               "Node 4 (1,1)", "Node 5 (0,2)", "Node 6 (1,2)"]
dof_labels   = ["u", "v", "θ"]

for i, label in enumerate(node_labels):
    u, v, theta = a[3*i, 0], a[3*i+1, 0], a[3*i+2, 0]
    print(f"  {label}: u={u:.6f}  v={v:.6f}  θ={theta:.6f}")

---
# Summary — CALFEM for Python

## What You Have Learned

This notebook introduced the **CALFEM** finite element library through four progressively complex examples:

| Example | Element type | Key concept |
|---|---|---|
| 1 — Spring system | `spring1e / spring1s` | FE workflow, assembly, boundary conditions |
| 2 — Three bars | `bar2e / bar2s` | 2D bar elements, coordinate transformation |
| 3 — 10-bar truss | `bar2e / bar2s` | Loop-based assembly for larger systems |
| 6 — Frame + bars | `beam2e / bar2e` | Mixed elements, DOF tables, `coordxtr` |

## Core Workflow (applies to every FE problem)

1. **Define topology** — `Edof` (or `edof1`/`edof2`): which DOFs belong to each element
2. **Define geometry** — `ex`, `ey`: end-node coordinates for each element
3. **Assemble** — loop over elements, call `cfc.assem(eltopo, K, Ke)` for each
4. **Apply loads and BCs** — fill `f` vector, set `bc` array
5. **Solve** — `a, r = cfc.solveq(K, f, bc)`
6. **Post-process** — `ed = cfc.extract_eldisp(Edof, a)`, then use section-force functions

## Key CALFEM Functions

| Function | Purpose |
|---|---|
| `spring1e(k)` | 1D spring element stiffness (2×2) |
| `spring1s(k, ed)` | Spring section force |
| `bar2e(ex, ey, ep)` | 2D bar element stiffness (4×4) |
| `bar2s(ex, ey, ep, ed)` | Bar axial force N |
| `beam2e(ex, ey, ep)` | 2D beam element stiffness (6×6) |
| `assem(edof, K, Ke)` | Assemble element K into global K |
| `solveq(K, f, bc)` | Solve with boundary conditions |
| `extract_eldisp(Edof, a)` | Extract element displacement vectors |
| `coordxtr(edof, coord, dof)` | Build ex/ey from node table |

## Further Reading

- [CALFEM for Python Manual](https://calfem-python-manual.readthedocs.io/en/latest/)
- [Element Functions Reference](https://calfem-python-manual.readthedocs.io/en/latest/element_functions.html)
- [System Functions Reference](https://calfem-python-manual.readthedocs.io/en/latest/system_functions.html)
- [Examples](https://calfem-python-manual.readthedocs.io/en/latest/examples.html)